In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms
import pandas as pd
import numpy as np
import SimpleITK as sitk
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
import timm
from tqdm import tqdm
import matplotlib.pyplot as plt
import os
from NoduleDS import NoduleDataset
from torch.optim.lr_scheduler import OneCycleLR
import torch.nn.functional as F
import cv2

# Set random seeds for reproducibility
torch.manual_seed(42)
np.random.seed(42)

# Device configuration
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

c:\Users\Edrill-LT\Documents\Projects\Python\Thoracic-Disease-Classifier-ResNet50\.torch\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Using device: cuda


In [2]:
BATCH_SIZE = 16
NUM_EPOCHS = 25
LEARNING_RATE = 1e-4
IMAGE_SIZE = 224
NUM_WORKERS = 4
WEIGHT_DECAY = 0.01

# Paths - using pre-split data to avoid leakage
split_base_path = "./dataset_nodule21/cxr_images/proccessed_data/split_data"

train_images_path = os.path.join(split_base_path, "train", "images")
val_images_path = os.path.join(split_base_path, "val", "images")
test_images_path = os.path.join(split_base_path, "test", "images")

train_csv_path = os.path.join(split_base_path, "train", "metadata_no_aug.csv")
val_csv_path = os.path.join(split_base_path, "val", "metadata_val.csv")
test_csv_path = os.path.join(split_base_path, "test", "metadata_test.csv")

train_df = pd.read_csv(train_csv_path)
val_df = pd.read_csv(val_csv_path)
test_df = pd.read_csv(test_csv_path)

In [3]:
train_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    #transforms.RandomHorizontalFlip(p=0.5),
    #transforms.RandomRotation(degrees=3),
    transforms.ColorJitter(brightness=0.15, contrast=0.15),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

val_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

In [4]:
# Create datasets - each pointing to its own split directory
train_dataset = NoduleDataset(train_df, train_images_path, transform=train_transform, return_bbox=True)
val_dataset = NoduleDataset(val_df, val_images_path, transform=val_transform, return_bbox=True)
test_dataset = NoduleDataset(test_df, test_images_path, transform=val_transform, return_bbox=True)

# Create dataloaders
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, 
                         num_workers=NUM_WORKERS, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, 
                       num_workers=NUM_WORKERS, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, 
                        num_workers=NUM_WORKERS, pin_memory=True)

In [5]:
# %% Calculate class weights for imbalanced dataset
print("\nCalculating class weights for imbalanced dataset...")

# Count class distribution in training set
train_labels = train_df['label'].values
class_counts = np.bincount(train_labels)
print(f"Training set distribution:")
print(f"  Class 0 (No Nodule): {class_counts[0]} samples")
print(f"  Class 1 (Nodule): {class_counts[1]} samples")

# Calculate weights: inverse of class frequency
total_samples = len(train_labels)
class_weights = total_samples / (len(class_counts) * class_counts)
class_weights = torch.FloatTensor(class_weights).to(device)

print(f"\nCalculated class weights:")
print(f"  Class 0 weight: {class_weights[0]:.4f}")
print(f"  Class 1 weight: {class_weights[1]:.4f}")


Calculating class weights for imbalanced dataset...
Training set distribution:
  Class 0 (No Nodule): 2623 samples
  Class 1 (Nodule): 1034 samples

Calculated class weights:
  Class 0 weight: 0.6971
  Class 1 weight: 1.7684
  Class 0 weight: 0.6971
  Class 1 weight: 1.7684


In [6]:
print("\nInitializing ResNet-50 model for TRUE WSOD...")
model = timm.create_model('resnet50', pretrained=False, num_classes=2)

# Load pre-trained checkpoint
checkpoint_path = './resnet50_binary.pth'
if os.path.exists(checkpoint_path):
    print(f"Loading checkpoint from {checkpoint_path}...")
    checkpoint = torch.load(checkpoint_path, map_location=device)
    model.load_state_dict(checkpoint['model_state_dict'])
    print("✓ Checkpoint loaded successfully")
else:
    print(f"Warning: Checkpoint {checkpoint_path} not found. Using randomly initialized model.")

model = model.to(device)

# WSOD Model with CAM generation capabilities
class TrueWSODResNet(nn.Module):
    def __init__(self, base_model):
        super().__init__()
        self.features = nn.Sequential(*list(base_model.children())[:-2])
        self.gap = nn.AdaptiveAvgPool2d(1)
        self.fc = base_model.fc if hasattr(base_model, 'fc') else base_model.get_classifier()
        
        # Get number of channels for CAM generation
        self.num_features = self.fc.in_features
        
    def forward(self, x, return_cam=False):
        features = self.features(x)  # [B, C, H, W]
        pooled = self.gap(features).flatten(1)  # [B, C]
        logits = self.fc(pooled)  # [B, num_classes]
        
        if return_cam:
            # Generate CAM during forward pass for training
            batch_size = features.size(0)
            cams = []
            
            for b in range(batch_size):
                # Use class weights to generate CAM
                weights = self.fc.weight[1]  # Weights for nodule class (class 1)
                cam = (features[b] * weights.view(-1, 1, 1)).sum(0)
                cam = F.relu(cam)
                cams.append(cam)
            
            cams = torch.stack(cams)  # [B, H, W]
            return logits, features, cams
        
        return logits, features

# Wrap model for WSOD
model = TrueWSODResNet(model)
model = model.to(device)

# WSOD-specific hyperparameters
LAMBDA_AREA = 0.1  # Weight for area constraint loss
LAMBDA_CENTER = 0.05  # Weight for center loss (encourages compact regions)

# Loss and optimizer
criterion_cls = nn.CrossEntropyLoss(weight=class_weights)
optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS)

print(f"Total trainable parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")
print("\n" + "="*60)
print("TRUE WSOD MODE: Training with WSOD-specific losses")
print("- Classification Loss (image-level labels)")
print("- Area Constraint Loss (compact attention)")
print("- Center Loss (focused localization)")
print("Bounding boxes are NOT used during training!")
print("="*60)


Initializing ResNet-50 model for TRUE WSOD...
Loading checkpoint from ./resnet50_binary.pth...
Loading checkpoint from ./resnet50_binary.pth...
✓ Checkpoint loaded successfully
Total trainable parameters: 23,512,130

TRUE WSOD MODE: Training with WSOD-specific losses
- Classification Loss (image-level labels)
- Area Constraint Loss (compact attention)
- Center Loss (focused localization)
Bounding boxes are NOT used during training!
✓ Checkpoint loaded successfully
Total trainable parameters: 23,512,130

TRUE WSOD MODE: Training with WSOD-specific losses
- Classification Loss (image-level labels)
- Area Constraint Loss (compact attention)
- Center Loss (focused localization)
Bounding boxes are NOT used during training!


In [7]:
# %% WSOD-Specific Loss Functions
def compute_area_loss(cams):
    """
    Area constraint loss: encourages compact attention regions
    Minimizes the area of high-attention regions
    """
    # Normalize CAMs to [0, 1]
    cams_norm = []
    for cam in cams:
        cam_min = cam.min()
        cam_max = cam.max()
        if cam_max > cam_min:
            cam_normalized = (cam - cam_min) / (cam_max - cam_min)
        else:
            cam_normalized = cam
        cams_norm.append(cam_normalized)
    
    cams_norm = torch.stack(cams_norm)
    
    # Threshold at 0.5 and compute area
    threshold = 0.5
    binary_maps = (cams_norm > threshold).float()
    area = binary_maps.sum(dim=(1, 2))  # Sum over H, W
    
    # Normalize by total pixels
    total_pixels = cams_norm.shape[1] * cams_norm.shape[2]
    area_ratio = area / total_pixels
    
    # Minimize area (encourage compact regions)
    return area_ratio.mean()

def compute_center_loss(cams):
    """
    Center loss: encourages attention to be concentrated in center regions
    This helps avoid scattered attention
    """
    batch_size, h, w = cams.shape
    
    # Create center weight mask (higher weight in center)
    center_y, center_x = h // 2, w // 2
    y_coords = torch.arange(h, device=cams.device).view(-1, 1).expand(h, w)
    x_coords = torch.arange(w, device=cams.device).view(1, -1).expand(h, w)
    
    # Distance from center
    dist = torch.sqrt(((y_coords - center_y) ** 2 + (x_coords - center_x) ** 2).float())
    dist_norm = dist / dist.max()
    
    # Weight mask (higher in center)
    center_weight = 1 - dist_norm
    
    # Weighted CAM sum (we want high values in center)
    weighted_cam = cams * center_weight.unsqueeze(0)
    
    # Maximize center attention (minimize negative)
    return -weighted_cam.mean()

def compute_wsod_loss(logits, labels, cams):
    """
    Combined WSOD loss with multiple components
    """
    # 1. Classification loss (standard cross-entropy)
    loss_cls = criterion_cls(logits, labels)
    
    # 2. Area constraint loss (only for positive samples)
    positive_mask = (labels == 1)
    if positive_mask.sum() > 0:
        positive_cams = cams[positive_mask]
        loss_area = compute_area_loss(positive_cams)
        loss_center = compute_center_loss(positive_cams)
    else:
        loss_area = torch.tensor(0.0, device=logits.device)
        loss_center = torch.tensor(0.0, device=logits.device)
    
    # 3. Total loss
    total_loss = loss_cls + LAMBDA_AREA * loss_area + LAMBDA_CENTER * loss_center
    
    return total_loss, loss_cls, loss_area, loss_center

In [8]:
# %% Training Functions for TRUE WSOD
from sklearn.metrics import precision_score, recall_score, f1_score, roc_auc_score

def train_epoch(model, loader, optimizer, device):
    model.train()
    running_loss = 0.0
    running_cls_loss = 0.0
    running_area_loss = 0.0
    running_center_loss = 0.0
    correct = 0
    total = 0
    all_preds = []
    all_labels = []
    all_probs = []
    
    pbar = tqdm(loader, desc='Training')
    for images, labels in pbar:
        images, labels = images.to(device), labels.to(device)
        
        optimizer.zero_grad()
        
        # Forward pass with CAM generation
        logits, features, cams = model(images, return_cam=True)
        
        # WSOD loss with multiple components
        loss, loss_cls, loss_area, loss_center = compute_wsod_loss(logits, labels, cams)
        
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item()
        running_cls_loss += loss_cls.item()
        running_area_loss += loss_area.item()
        running_center_loss += loss_center.item()
        
        probs = torch.softmax(logits, dim=1)
        _, predicted = logits.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()
        
        all_preds.extend(predicted.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
        all_probs.extend(probs[:, 1].cpu().detach().numpy())
        
        pbar.set_postfix({
            'loss': running_loss/len(pbar), 
            'cls': running_cls_loss/len(pbar),
            'area': running_area_loss/len(pbar),
            'acc': 100.*correct/total
        })
    
    epoch_loss = running_loss / len(loader)
    epoch_acc = 100. * correct / total
    
    return epoch_loss, epoch_acc, all_preds, all_labels, all_probs, {
        'cls_loss': running_cls_loss / len(loader),
        'area_loss': running_area_loss / len(loader),
        'center_loss': running_center_loss / len(loader)
    }

def validate(model, loader, device):
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0
    all_preds = []
    all_labels = []
    all_probs = []
    
    with torch.no_grad():
        for images, labels in tqdm(loader, desc='Validation'):
            images, labels = images.to(device), labels.to(device)
            
            logits, features, cams = model(images, return_cam=True)
            loss, loss_cls, loss_area, loss_center = compute_wsod_loss(logits, labels, cams)
            
            running_loss += loss.item()
            probs = torch.softmax(logits, dim=1)
            _, predicted = logits.max(1)
            
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()
            
            all_preds.extend(predicted.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
            all_probs.extend(probs[:, 1].cpu().numpy())

    epoch_loss = running_loss / len(loader)
    epoch_acc = 100. * correct / total
    
    # Binary metrics
    binary_precision = precision_score(all_labels, all_preds, average='binary', zero_division=0)
    binary_recall = recall_score(all_labels, all_preds, average='binary', zero_division=0)
    binary_f1 = f1_score(all_labels, all_preds, average='binary', zero_division=0)
    binary_auc = roc_auc_score(all_labels, all_probs)
    
    return epoch_loss, epoch_acc, all_preds, all_labels, all_probs, binary_precision, binary_recall, binary_f1, binary_auc

In [9]:
# %% Training Loop for TRUE WSOD
print("\nStarting TRUE WSOD training...")
history = {
    'train_loss': [], 'train_acc': [], 'train_auc': [], 'train_precision': [],
    'train_cls_loss': [], 'train_area_loss': [], 'train_center_loss': [],
    'val_loss': [], 'val_acc': [], 'val_auc': [], 'val_precision': []
}

best_val_acc = 0.0
best_model_path = 'best_wsod_resnet_model.pth'

for epoch in range(NUM_EPOCHS):
    print(f"\nEpoch {epoch+1}/{NUM_EPOCHS}")
    print("-" * 70)
    
    # Train
    train_loss, train_acc, train_preds, train_labels, train_probs, train_losses = train_epoch(
        model, train_loader, optimizer, device)
    
    # Calculate train metrics
    train_auc = roc_auc_score(train_labels, train_probs)
    train_precision = precision_score(train_labels, train_preds, average='binary', zero_division=0)
    
    # Validate
    val_loss, val_acc, val_preds, val_labels, val_probs, val_precision, val_recall, val_f1, val_auc = validate(
        model, val_loader, device)
    
    # Step scheduler
    scheduler.step()
    
    # Save history
    history['train_loss'].append(train_loss)
    history['train_acc'].append(train_acc)
    history['train_auc'].append(train_auc)
    history['train_precision'].append(train_precision)
    history['train_cls_loss'].append(train_losses['cls_loss'])
    history['train_area_loss'].append(train_losses['area_loss'])
    history['train_center_loss'].append(train_losses['center_loss'])
    history['val_loss'].append(val_loss)
    history['val_acc'].append(val_acc)
    history['val_auc'].append(val_auc)
    history['val_precision'].append(val_precision)
    
    print(f"Train - Total: {train_loss:.4f}, Cls: {train_losses['cls_loss']:.4f}, "
          f"Area: {train_losses['area_loss']:.4f}, Center: {train_losses['center_loss']:.4f}")
    print(f"Train - Acc: {train_acc:.2f}%, AUC: {train_auc:.4f}, Precision: {train_precision:.4f}")
    print(f"Val   - Loss: {val_loss:.4f}, Acc: {val_acc:.2f}%, AUC: {val_auc:.4f}, Precision: {val_precision:.4f}")
    print(f"LR: {optimizer.param_groups[0]['lr']:.6f}")
    
    # Save best model
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'val_acc': val_acc,
            'val_auc': val_auc,
            'history': history
        }, best_model_path)
        print(f"✓ Saved best WSOD model (Val Acc: {val_acc:.2f}%, Val AUC: {val_auc:.4f})")

print("\n" + "="*70)
print("TRUE WSOD Training completed!")
print("Model learned to localize using ONLY image-level labels!")
print("="*70)


Starting TRUE WSOD training...

Epoch 1/25
----------------------------------------------------------------------


Training:   0%|          | 0/229 [00:11<?, ?it/s]



ValueError: too many values to unpack (expected 2)

In [ ]:
# %% Plot Training History with WSOD Losses
fig, axes = plt.subplots(2, 3, figsize=(18, 12))

# Total Loss
axes[0, 0].plot(history['train_loss'], label='Train Loss', linewidth=2)
axes[0, 0].plot(history['val_loss'], label='Val Loss', linewidth=2)
axes[0, 0].set_xlabel('Epoch', fontsize=12)
axes[0, 0].set_ylabel('Total Loss', fontsize=12)
axes[0, 0].set_title('Total WSOD Loss', fontsize=14, fontweight='bold')
axes[0, 0].legend(fontsize=11)
axes[0, 0].grid(True, alpha=0.3)

# Accuracy
axes[0, 1].plot(history['train_acc'], label='Train Acc', linewidth=2)
axes[0, 1].plot(history['val_acc'], label='Val Acc', linewidth=2)
axes[0, 1].set_xlabel('Epoch', fontsize=12)
axes[0, 1].set_ylabel('Accuracy (%)', fontsize=12)
axes[0, 1].set_title('Accuracy', fontsize=14, fontweight='bold')
axes[0, 1].legend(fontsize=11)
axes[0, 1].grid(True, alpha=0.3)

# AUC
axes[0, 2].plot(history['train_auc'], label='Train AUC', linewidth=2)
axes[0, 2].plot(history['val_auc'], label='Val AUC', linewidth=2)
axes[0, 2].set_xlabel('Epoch', fontsize=12)
axes[0, 2].set_ylabel('AUC', fontsize=12)
axes[0, 2].set_title('ROC AUC', fontsize=14, fontweight='bold')
axes[0, 2].legend(fontsize=11)
axes[0, 2].grid(True, alpha=0.3)

# Classification Loss
axes[1, 0].plot(history['train_cls_loss'], label='Classification Loss', linewidth=2, color='orange')
axes[1, 0].set_xlabel('Epoch', fontsize=12)
axes[1, 0].set_ylabel('Loss', fontsize=12)
axes[1, 0].set_title('Classification Loss', fontsize=14, fontweight='bold')
axes[1, 0].legend(fontsize=11)
axes[1, 0].grid(True, alpha=0.3)

# Area Constraint Loss
axes[1, 1].plot(history['train_area_loss'], label='Area Loss', linewidth=2, color='green')
axes[1, 1].set_xlabel('Epoch', fontsize=12)
axes[1, 1].set_ylabel('Loss', fontsize=12)
axes[1, 1].set_title('Area Constraint Loss (WSOD)', fontsize=14, fontweight='bold')
axes[1, 1].legend(fontsize=11)
axes[1, 1].grid(True, alpha=0.3)

# Center Loss
axes[1, 2].plot(history['train_center_loss'], label='Center Loss', linewidth=2, color='red')
axes[1, 2].set_xlabel('Epoch', fontsize=12)
axes[1, 2].set_ylabel('Loss', fontsize=12)
axes[1, 2].set_title('Center Loss (WSOD)', fontsize=14, fontweight='bold')
axes[1, 2].legend(fontsize=11)
axes[1, 2].grid(True, alpha=0.3)

plt.suptitle('TRUE WSOD Training History', fontsize=16, fontweight='bold', y=1.00)
plt.tight_layout()
plt.savefig('wsod_training_history.png', dpi=150)
plt.show()

In [ ]:
# %% Load Best Model and Evaluate
print("\nLoading best WSOD model for testing...")
checkpoint = torch.load(best_model_path)
model.load_state_dict(checkpoint['model_state_dict'])

test_loss, test_acc, test_preds, test_labels, test_probs, test_precision, test_recall, test_f1, test_auc = validate(
    model, test_loader, device)

print(f"\n" + "="*70)
print("TRUE WSOD Test Results")
print("="*70)
print(f"Test Loss: {test_loss:.4f}")
print(f"Test Accuracy: {test_acc:.2f}%")
print(f"Test Precision: {test_precision:.4f}")
print(f"Test Recall: {test_recall:.4f}")
print(f"Test F1 Score: {test_f1:.4f}")
print(f"Test ROC AUC: {test_auc:.4f}")

print("\nClassification Report:")
print(classification_report(test_labels, test_preds, target_names=['No Nodule', 'Nodule']))
print("="*70)

In [ ]:
# %% Confusion Matrix
from sklearn.metrics import confusion_matrix
import seaborn as sns

cm = confusion_matrix(test_labels, test_preds)
cm_labels = ['No Nodule', 'Nodule']

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=cm_labels, yticklabels=cm_labels, cbar_kws={'label': 'Count'})
plt.xlabel('Predicted Label', fontsize=12)
plt.ylabel('True Label', fontsize=12)
plt.title('Confusion Matrix - TRUE WSOD Model', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('wsod_confusion_matrix.png', dpi=150)
plt.show()
print("✓ Saved: wsod_confusion_matrix.png")

In [ ]:
# %% CAM Extractor for Visualization
class WSODCAMExtractor:
    """Extract CAMs from trained WSOD model"""
    def __init__(self, model):
        self.model = model
    
    def generate_cam(self, input_image, target_class=1):
        """Generate CAM for visualization"""
        self.model.eval()
        
        with torch.no_grad():
            logits, features, cams = self.model(input_image, return_cam=True)
        
        # Return normalized CAM
        cam = cams[0].cpu().numpy()  # First image in batch
        
        # Normalize to [0, 1]
        cam = cam - cam.min()
        if cam.max() > 0:
            cam = cam / cam.max()
        
        return cam

# Initialize CAM extractor
print("\nInitializing WSOD CAM extractor for visualization...")
cam_extractor = WSODCAMExtractor(model)
print("✓ WSOD CAM extractor initialized")
print("✓ CAMs from model trained with area and center constraints!")